In [ ]:
"""
BigAlpha 2026 提交代码 —— ITFP 因子 (日内成交碎片化压力)
=====================================================
"""

import pandas as pd
import numpy as np
import dai


# ═══════════════════════════════════════════════════════════════════════════
# 因子参数
# ═══════════════════════════════════════════════════════════════════════════
N_PERIODS = 24          # 10分钟窗口数 (09:30-11:30→0-11, 13:00-15:00→12-23)
SMOOTH_WINDOW = 60      # 时序平滑窗口(交易日)
EPS = 1e-12             # 数值稳定小量

# SMA60 需要 60 交易日历史 ≈ 90 自然日, 给 120 宁多勿少
BUFFER_DAYS = 120


# ═══════════════════════════════════════════════════════════════════════════
# 主函数 (比赛入口)
# ═══════════════════════════════════════════════════════════════════════════

def main(datasources, start_date, end_date):
    """
    比赛要求入口函数。

    Args:
        datasources: dict, 至少包含 {'bar1m': '<分钟量表>', 'financial': '<财务表>'}
        start_date:  str, 评估区间起始时间
        end_date:    str, 评估区间结束时间

    Returns:
        pd.DataFrame, 三列 ['date', 'instrument', 'factor']
    """
    # ✅ 表名从 datasources 取
    bar1m = datasources["bar1m"]

    # ✅ 向前扩查询窗口, 给 SMA60 时序平滑喂历史
    query_start = pd.to_datetime(start_date) - pd.Timedelta(days=BUFFER_DAYS)

    # ═══════════════════════════════════════════════════════════════════
    # 步骤1: DAI SQL 10分钟窗口聚合 (分钟级 → 窗口级长表)
    # ═══════════════════════════════════════════════════════════════════
    #
    # base:     分钟数据 + period 划分 (10分钟窗口)
    # windowed: row_number 标记每个窗口的首/尾分钟
    # agg:      按 (date, instrument, period) 聚合:
    #           - p_herf  = Σ(volume²) / (Σvolume)²    Herfindahl 集中度
    #           - p_open  = 窗口首分钟开盘价
    #           - p_close = 窗口末分钟收盘价
    #
    # period 划分:
    #   上午 09:30-11:30 → period 0-11 (每10分钟一个)
    #   下午 13:00-15:00 → period 12-23
    #
    # argMax/argMin 替换: 用 row_number + CASE WHEN 取首/尾值

    sql = f"""
    WITH base AS (
        SELECT
            instrument,
            CAST(date AS DATE) AS trade_date,
            date,
            close::DOUBLE AS close,
            "open"::DOUBLE AS open,
            volume::DOUBLE AS volume,
            CASE
                WHEN extract(hour from date) < 12
                THEN ((extract(hour from date) - 9) * 60 + extract(minute from date) - 30) // 10
                ELSE ((extract(hour from date) - 13) * 60 + extract(minute from date) + 120) // 10
            END AS period
        FROM {bar1m}
        WHERE close > 0
          AND volume > 0
    ),
    windowed AS (
        SELECT
            instrument,
            trade_date,
            period,
            volume,
            volume * volume AS vol_sq,
            close,
            open,
            row_number() OVER (
                PARTITION BY instrument, trade_date, period
                ORDER BY date
            ) AS rn_asc,
            row_number() OVER (
                PARTITION BY instrument, trade_date, period
                ORDER BY date DESC
            ) AS rn_desc
        FROM base
        WHERE period BETWEEN 0 AND {N_PERIODS - 1}
    )
    SELECT
        CAST(trade_date AS TIMESTAMP) AS date,
        instrument,
        period,
        sum(vol_sq) / (sum(volume) * sum(volume) + {EPS}) AS p_herf,
        max(CASE WHEN rn_asc = 1 THEN open END) AS p_open,
        max(CASE WHEN rn_desc = 1 THEN close END) AS p_close
    FROM windowed
    GROUP BY trade_date, instrument, period
    ORDER BY trade_date, instrument, period
    """

    df_long = dai.query(
        sql,
        filters={
            'date': [
                query_start.strftime('%Y-%m-%d %H:%M:%S'),
                end_date,
            ]
        },
        compression=True,
    ).df()

    if df_long.empty:
        return pd.DataFrame(columns=['date', 'instrument', 'factor'])

    # ═══════════════════════════════════════════════════════════════════
    # 步骤2: pandas pivot 长表 → 宽表 (24列 × 3组)
    # ═══════════════════════════════════════════════════════════════════

    # 透视: (date, instrument) × period → 各指标列
    df_herf = df_long.pivot_table(
        index=['date', 'instrument'], columns='period',
        values='p_herf', aggfunc='first'
    )
    df_open = df_long.pivot_table(
        index=['date', 'instrument'], columns='period',
        values='p_open', aggfunc='first'
    )
    df_close = df_long.pivot_table(
        index=['date', 'instrument'], columns='period',
        values='p_close', aggfunc='first'
    )

    # 重命名列: p_herf_0, p_herf_1, ..., p_herf_23
    df_herf.columns = [f'p_herf_{c}' for c in df_herf.columns]
    df_open.columns = [f'p_open_{c}' for c in df_open.columns]
    df_close.columns = [f'p_close_{c}' for c in df_close.columns]

    # 合并
    df_wide = df_herf.join(df_open).join(df_close).reset_index()

    # 补齐缺失的 period 列 (某些日期可能缺少部分窗口)
    for i in range(N_PERIODS):
        for prefix in ['p_herf', 'p_open', 'p_close']:
            col = f'{prefix}_{i}'
            if col not in df_wide.columns:
                df_wide[col] = np.nan

    # ═══════════════════════════════════════════════════════════════════
    # 步骤3: numpy 向量化计算 ITFP 因子
    # ═══════════════════════════════════════════════════════════════════

    df_wide = df_wide.sort_values(['instrument', 'date']).reset_index(drop=True)

    herf_cols  = [f'p_herf_{i}'  for i in range(N_PERIODS)]
    close_cols = [f'p_close_{i}' for i in range(N_PERIODS)]
    open_cols  = [f'p_open_{i}'  for i in range(N_PERIODS)]

    herf    = df_wide[herf_cols].values.astype(np.float64)
    close_p = df_wide[close_cols].values.astype(np.float64)
    open_p  = df_wide[open_cols].values.astype(np.float64)

    # 有效窗口掩码
    valid = (~np.isnan(herf)) & (herf > 0)

    # 1. 自身归一化 (当日 24 窗口间, 不跨股票)
    with np.errstate(all='ignore'):
        herf_min = np.nanmin(np.where(valid, herf, np.nan), axis=1, keepdims=True)
        herf_max = np.nanmax(np.where(valid, herf, np.nan), axis=1, keepdims=True)
        herf_norm = (herf - herf_min) / (herf_max - herf_min + EPS)

    # 2. 窗口收益率
    close_safe = np.where(valid, close_p, np.nan)
    open_safe  = np.where(valid, open_p,  np.nan)

    ret = np.full_like(close_p, np.nan)
    # 首窗口: close[0]/open[0] - 1
    ret[:, 0] = close_safe[:, 0] / (open_safe[:, 0] + EPS) - 1
    # 后续窗口: close[i]/close[i-1] - 1
    prev_close = close_safe[:, :-1]
    curr_close = close_safe[:, 1:]
    prev_valid = valid[:, :-1]
    curr_valid = valid[:, 1:]

    with np.errstate(all='ignore'):
        ret[:, 1:] = np.where(
            curr_valid,
            np.where(prev_valid,
                     curr_close / (prev_close + EPS) - 1,
                     curr_close / (open_safe[:, 1:] + EPS) - 1),
            np.nan
        )

    # 3. Pressure = ret × herf_norm
    with np.errstate(all='ignore'):
        pressure = ret * herf_norm

    # 4. ITFP = 累加和 (nansum, 无成交量加权)
    itfp = np.nansum(pressure, axis=1)
    n_valid = valid.sum(axis=1)
    itfp = np.where(n_valid > 0, itfp, np.nan)

    df_wide['itfp'] = itfp

    # ═══════════════════════════════════════════════════════════════════
    # 步骤4: pandas 后处理 (去极值 → cs_rank → 翻转 → SMA60)
    # ═══════════════════════════════════════════════════════════════════

    # 清洗
    df_wide['itfp'] = df_wide['itfp'].replace([np.inf, -np.inf], np.nan)

    # 截面去极值 (0.5% ~ 99.5%)
    def winsorize(s):
        lo, hi = s.quantile(0.005), s.quantile(0.995)
        return s.clip(lo, hi)

    df_wide['itfp_clipped'] = df_wide.groupby('date')['itfp'].transform(winsorize)

    # 截面 pct rank → 翻转符号 (1 - rank)
    df_wide['itfp_rank'] = df_wide.groupby('date')['itfp_clipped'].rank(pct=True)
    df_wide['itfp_rank_neg'] = 1.0 - df_wide['itfp_rank']

    # SMA60 时序平滑 (按股票分组)
    df_wide['factor'] = df_wide.groupby('instrument')['itfp_rank_neg'].transform(
        lambda x: x.rolling(
            window=SMOOTH_WINDOW,
            min_periods=min(SMOOTH_WINDOW, 3),
        ).mean()
    )

    # ═══════════════════════════════════════════════════════════════════
    # 步骤5: 裁回评估区间 + 过滤成分股 + 返回
    # ═══════════════════════════════════════════════════════════════════

    # ✅ 裁回真正的评估区间
    start_ts = pd.to_datetime(start_date)
    end_ts = pd.to_datetime(end_date)
    df_wide = df_wide[(df_wide['date'] >= start_ts) & (df_wide['date'] <= end_ts)]

    # ✅ 过滤到中证1000成分股
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]},
    ).df()

    df_wide = pd.merge(df_wide, stk_pool, how='inner', on=['date', 'instrument'])

    # ── 缺失值处理: 仅 ffill, 严禁 bfill (未来函数) ──
    df_wide = df_wide.sort_values(['instrument', 'date'])
    df_wide['factor'] = df_wide.groupby('instrument')['factor'].ffill()
    df_wide['factor'] = df_wide['factor'].replace([np.inf, -np.inf], np.nan)
    df_wide['factor'] = df_wide['factor'].fillna(0.0)

    return df_wide[['date', 'instrument', 'factor']]


# ═══════════════════════════════════════════════════════════════════════════
# 本地自检代码 (提交前务必跑一遍)
# ═══════════════════════════════════════════════════════════════════════════

def selftest_coverage():
    """自检一: 覆盖度检查"""
    print("=" * 60)
    print("自检一: 覆盖度检查")
    print("=" * 60)

    datasources = {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'}
    start_date = '2024-01-01 00:00:00'
    end_date = '2024-10-31 23:59:59'

    factor_data = main(datasources, start_date, end_date)

    print(f"  总行数: {len(factor_data):,}")
    print(f"  日期数: {factor_data['date'].nunique()}")
    print(f"  股票数: {factor_data['instrument'].nunique()}")
    print(f"  NaN数:  {factor_data['factor'].isna().sum()}")

    for d in ['2024-01-02', '2024-01-03', '2024-01-04', '2024-01-05']:
        day_data = factor_data[factor_data['date'] == pd.to_datetime(d)]
        if len(day_data) == 0:
            print(f"  {d}: 无数据")
            continue
        nan_rate = day_data['factor'].isna().mean()
        print(f"  {d}: 缺失率 = {nan_rate:.2%}")
        assert nan_rate < 0.4, f"❌ 覆盖度不足: {d} 缺失率 {nan_rate:.2%}"

    print("  ✅ 覆盖度自检通过\n")


def selftest_lookahead():
    """自检二: 未来函数检查"""
    print("=" * 60)
    print("自检二: 未来函数检查")
    print("=" * 60)

    start_date = '2024-01-01 00:00:00'
    end_date = '2024-01-31 23:59:59'

    df_full = main(
        {'bar1m': 'bigalpha_2026_stock_bar1m_selftest'},
        start_date, end_date,
    )
    df_cut = main(
        {'bar1m': 'bigalpha_2026_stock_bar1m_ahead'},
        start_date, end_date,
    )

    cutoff = '2024-01-31 23:59:59'
    df_full_b = df_full[df_full['date'] <= pd.to_datetime(cutoff)]
    df_cut_b = df_cut[df_cut['date'] <= pd.to_datetime(cutoff)]

    merged = pd.merge(
        df_full_b, df_cut_b,
        on=['date', 'instrument'],
        suffixes=('_full', '_cut'),
    )
    diff = (merged['factor_full'] - merged['factor_cut']).abs()
    bad = merged[diff > 1e-5]

    if len(bad) > 0:
        print(f"  ❌ 发现未来函数, 差异行数: {len(bad)}")
        print(bad.head(10))
    else:
        print("  ✅ 未来函数自检通过")


if __name__ == '__main__':
    selftest_coverage()
    selftest_lookahead()
